In [ ]:
import sys
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import LeaveOneOut
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from code.Data_Preprocessing import DataPreprocessing_melt as mf
from code.Data_Processing import TargetVariables as tv
from code.Data_Processing import DataProcessing_run as rf
import os

In [ ]:
encoder = LabelEncoder()
lower_pct = 20
upper_pct = 80
MAX_HDRS = 69 
MAX_CDI = 90  
# Load EMB35.pkl file 
df = pd.read_pickle("/planilhas/EMB35.pkl")

- Note: For calculating the standardized Y values using HDRS/CDI raw score data, refer to the worksheet in the data folder of this project's main DOI. [Use "Patient_ID" as a cross-reference.]

___

M2 - YAA (S3) (Y = Delta_Y : Worse and Better)

In [ ]:
df_model_YAA_S3_M = df.copy()
df_model_YAA_S3_M = tv.standardized_classify_evolution(df_model_YAA_S3_M, MAX_HDRS, MAX_CDI, lower_pct, upper_pct)
df_model_YAA_S3_M['Y_Classe_Evolution_Delta_Y'] = df_model_YAA_S3_M['Y_Classe_Evolution_Delta_Y'].map({'Worse': 0, 'Stable': 1, 'Better': 2})
df_model_YAA_S3_M = df_model_YAA_S3_M.dropna(subset=['Y_Classe_Evolution_Delta_Y'])
df_model_YAA_S3_M['Y_Classe_Evolution_Delta_Y'] = df_model_YAA_S3_M['Y_Classe_Evolution_Delta_Y'].astype(int)

In [ ]:
meta_colsYAA = ['Patient_ID', 'Y_Classe_Evolution_Delta_Y']
df_model_YAA_S3_M = mf.get_embeddings_per_segment_mean_std_2D(df_model_YAA_S3_M, meta_colsYAA)
df_model_YAA_S3_M = df_model_YAA_S3_M.dropna()
print(f"Patients: {df_model_YAA_S3_M['Patient_ID'].nunique()}")

____

M4 - YA (S3) (Y = Delta_HDRS : Worse and Better)

In [ ]:
df_model_YA_S3_M = df.copy()
df_model_YA_S3_M = tv.standardized_classify_evolution_HDRS(df_model_YA_S3_M, MAX_HDRS, MAX_CDI, lower_pct, upper_pct)
df_model_YA_S3_M['Y_Classe_Evolution_HDRS'] = df_model_YA_S3_M['Y_Classe_Evolution_HDRS'].map({'Worse': 0, 'Stable': 1, 'Better': 2})
df_model_YA_S3_M = df_model_YA_S3_M.dropna(subset=['Y_Classe_Evolution_HDRS'])
df_model_YA_S3_M['Y_Classe_Evolution_HDRS'] = df_model_YA_S3_M['Y_Classe_Evolution_HDRS'].astype(int)

In [ ]:
meta_colsYA = ['Patient_ID', 'Y_Classe_Evolution_HDRS']
df_model_YA_S3_M = mf.get_embeddings_per_segment_mean_std_2D(df_model_YA_S3_M, meta_colsYA)
df_model_YA_S3_M = df_model_YA_S3_M.dropna()
print(f"Patients: {df_model_YA_S3_M['Patient_ID'].nunique()}")

M6 - A (S3) (Y = Delta_CDI : Worse and Better)

In [ ]:
df_model_A_S3_M = df.copy()
df_model_A_S3_M = tv.standardized_classify_evolution_CDI(df_model_A_S3_M, MAX_HDRS, MAX_CDI, lower_pct, upper_pct)
df_model_A_S3_M['Y_Classe_Evolution_CDI'] = df_model_A_S3_M['Y_Classe_Evolution_CDI'].map({'Worse': 0, 'Stable': 1, 'Better': 2})
df_model_A_S3_M = df_model_A_S3_M.dropna(subset=['Y_Classe_Evolution_CDI'])
df_model_A_S3_M['Y_Classe_Evolution_CDI'] = df_model_A_S3_M['Y_Classe_Evolution_CDI'].astype(int)

In [ ]:
meta_colsA = ['Patient_ID', 'Y_Classe_Evolution_CDI']
df_model_A_S3_M = mf.get_embeddings_per_segment_mean_std_2D(df_model_A_S3_M, meta_colsA)
df_model_A_S3_M = df_model_A_S3_M.dropna()
print(f"Patients: {df_model_A_S3_M['Patient_ID'].nunique()}")

___

In [ ]:
datasets = [
    (df_model_YAA_S3_M, 'Y_Classe_Evolution_Delta_Y')
    (df_model_YA_S3_M, 'Y_Classe_Evolution_HDRS')
    (df_model_A_S3_M, 'Y_Classe_Evolution_CDI')]

In [ ]:
dataframes = [item[0] for item in datasets]
for df_index in dataframes:
    cols_with_nan = df_index.columns[df_index.isna().any()]
    print("Columns with None ou NaT:", cols_with_nan.tolist())

____

Running the experiments (LOO-CV patient-independent)

In [ ]:
summary_results = []
all_results = []
for idx, (df, target_column) in enumerate(datasets):
    print(f"\nProcessing Dataset {idx+1} - Target: {target_column}")
    unique_patients = df['Patient_ID'].unique()    
    results_rf = [rf.class_process_leave_one_out(patient, df, 'Patient_ID', target_column, RandomForestClassifier(class_weight='balanced', random_state=42), rf.class_param_grid_rf)
        for patient in unique_patients]
    print("RF Done")
    result_lr = [rf.class_process_leave_one_out(patient, df, 'Patient_ID', target_column, LogisticRegression(class_weight='balanced', random_state=42), rf.class_param_grid_logreg)
        for patient in unique_patients]
    print("LogR Done")
    results_xgb = [rf.class_process_leave_one_out(patient, df, 'Patient_ID', target_column, xgb.XGBClassifier(n_jobs=1, random_state=42, use_label_encoder=False, eval_metric='mlogloss', objective='multi:softprob'), rf.class_param_grid_xgb)
        for patient in unique_patients]
    print("XGB Done") 
    results_mlp = [rf.class_process_leave_one_out(patient, df, 'Patient_ID', target_column, MLPClassifier(random_state=42), rf.class_param_grid_mlp)
        for patient in unique_patients]
    print("MLP Done")
    current_results = results_rf + result_lr + results_xgb + results_mlp
    all_results.extend(current_results)   
    df_current = pd.DataFrame([r for r in current_results if r is not None])
    for model_name in df_current["Model"].unique():
        df_model = df_current[df_current["Model"] == model_name]
        summary_results.append({
            "Dataset_Index": idx + 1,
            "Target_Column": target_column,
            "Model": model_name,
            "Recall_Mean": df_model["Recall"].mean(),
            "F1_Score": df_model["F1_Score"].mean(),
            "Precision_Mean": df_model["Precision"].mean()
        })